<a href="https://colab.research.google.com/github/Khuongtdm/Rebuild-Yolov1-from-scratch/blob/main/mATHYolo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
def IOU(boxes_preds, boxes_labels, box_format="midpoint"):
    """
    Calculates intersection over union

    Parameters:
        boxes_preds (tensor): Predictions of Bounding Boxes (BATCH_SIZE, 4)
        boxes_labels (tensor): Correct labels of Bounding Boxes (BATCH_SIZE, 4)
        box_format (str): midpoint/corners, if boxes (x,y,w,h) or (x1,y1,x2,y2)

    Returns:
        tensor: Intersection over union for all examples
    """

    if box_format == "midpoint":
        box1_x1 = boxes_preds[..., 0:1] - boxes_preds[..., 2:3] / 2
        box1_y1 = boxes_preds[..., 1:2] - boxes_preds[..., 3:4] / 2
        box1_x2 = boxes_preds[..., 0:1] + boxes_preds[..., 2:3] / 2
        box1_y2 = boxes_preds[..., 1:2] + boxes_preds[..., 3:4] / 2
        box2_x1 = boxes_labels[..., 0:1] - boxes_labels[..., 2:3] / 2
        box2_y1 = boxes_labels[..., 1:2] - boxes_labels[..., 3:4] / 2
        box2_x2 = boxes_labels[..., 0:1] + boxes_labels[..., 2:3] / 2
        box2_y2 = boxes_labels[..., 1:2] + boxes_labels[..., 3:4] / 2

    if box_format == "corners":
        box1_x1 = boxes_preds[..., 0:1]
        box1_y1 = boxes_preds[..., 1:2]
        box1_x2 = boxes_preds[..., 2:3]
        box1_y2 = boxes_preds[..., 3:4]  # (N, 1)
        box2_x1 = boxes_labels[..., 0:1]
        box2_y1 = boxes_labels[..., 1:2]
        box2_x2 = boxes_labels[..., 2:3]
        box2_y2 = boxes_labels[..., 3:4]

    x1 = torch.max(box1_x1, box2_x1)
    y1 = torch.max(box1_y1, box2_y1)
    x2 = torch.min(box1_x2, box2_x2)
    y2 = torch.min(box1_y2, box2_y2)

    # .clamp(0) is for the case when they do not intersect
    intersection = (x2 - x1).clamp(0) * (y2 - y1).clamp(0)

    box1_area = abs((box1_x2 - box1_x1) * (box1_y2 - box1_y1))
    box2_area = abs((box2_x2 - box2_x1) * (box2_y2 - box2_y1))

    return intersection / (box1_area + box2_area - intersection + 1e-6)



In [ ]:
"""
some utilities to use that make my model work more properly
provided by Aladdin Persson

"""
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from collections import Counter
def save_checkpoint(state, filename="my_checkpoint.pth.tar"):
    print("=> Saving checkpoint")
    torch.save(state, filename)

def load_checkpoint(checkpoint, model, optimizer):
    print("=> Loading checkpoint")
    model.load_state_dict(checkpoint["state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer"])



def non_max_suppression(bboxes, iou_threshold, threshold, box_format="corners"):
    """
    Does Non Max Suppression given bboxes

    Parameters:
        bboxes (list): list of lists containing all bboxes with each bboxes
        specified as [class_pred, prob_score, x1, y1, x2, y2]
        iou_threshold (float): threshold where predicted bboxes is correct
        threshold (float): threshold to remove predicted bboxes (independent of IoU)
        box_format (str): "midpoint" or "corners" used to specify bboxes

    Returns:
        list: bboxes after performing NMS given a specific IoU threshold
    """

    assert type(bboxes) == list

    bboxes = [box for box in bboxes if box[1] > threshold]
    bboxes = sorted(bboxes, key=lambda x: x[1], reverse=True)
    bboxes_after_nms = []

    while bboxes:
        chosen_box = bboxes.pop(0)

        bboxes = [
            box
            for box in bboxes
            if box[0] != chosen_box[0]
            or IOU(
                torch.tensor(chosen_box[2:]),
                torch.tensor(box[2:]),
                box_format=box_format,
            )
            < iou_threshold
        ]

        bboxes_after_nms.append(chosen_box)

    return bboxes_after_nms
def plot_image(image, boxes):
    """Plots predicted bounding boxes on the image"""
    im = np.array(image)
    height, width, _ = im.shape

    # Create figure and axes
    fig, ax = plt.subplots(1)
    # Display the image
    ax.imshow(im)

    # box[0] is x midpoint, box[2] is width
    # box[1] is y midpoint, box[3] is height

    # Create a Rectangle potch
    for box in boxes:
        box = box[2:]
        assert len(box) == 4, "Got more values than in x, y, w, h, in a box!"
        upper_left_x = box[0] - box[2] / 2
        upper_left_y = box[1] - box[3] / 2
        rect = patches.Rectangle(
            (upper_left_x * width, upper_left_y * height),
            box[2] * width,
            box[3] * height,
            linewidth=1,
            edgecolor="r",
            facecolor="none",
        )
        # Add the patch to the Axes
        ax.add_patch(rect)

    plt.show()


In [ ]:
import torch
from collections import Counter
def mean_average_presision(pre_boxes,true_boxes,IOU_threshold,box_format="conner",num_classes=20):

  """
  theory of maP:
  every image after the process of prediction will have:
  - real bboxes: the actual bboxes given by the user dataset
  - predicted bboxes: the bboxes that created by AI
  - confidence score: AI confidence about that bboxes (just like when ur teacher ask to to predicted something and this is just your confident with your predicted)
  there will be 3 outcome: given the IOU threshold by the user:
  case 1: the bboxes by the AI is completely out from the actual bboxes: no matter how big the confidence score, the answer will be FP (false positive)
  case 2: the box by AI is inside the actual box but the IOU value is < IOU threshold, FP
  case 3: the box by AI is inside the actual box and IOU value >= IOU threshold, TP
 given 2 concept of estimation:
 precision and recall
 Consider the case where an imgae contents 2 actual bboxes and 3 prediction bboxes which has one bbox is out of the actual box and the rest 2 bboxes is in the actual box
 however only one prediction bbox in the actual bbox has theIOU value >= IOU threshold, TP => 2 FP and 1 TP
 recall = TP/ the number of prediction boxes that in the actual box = 1/2
 precicion TP/ the total of prediction boxes= 1/3
 then we take the average of (recall+precision)/2.
 However, the maP process consist of difference IOU threshold, each of them will yield difference average result, maP final result will be the total average of all of them
 for example: maP@0.5,0.6,0.95 will the the avergae of maP value when the IOU thresholds of each calculation are 0.5, 0.6 and 0.95 respectively.
 if we have multiple classes, the process will reapeat for each class

 """
  # pre_box (list) = [[traning_idx,class_pre,confidence prob_score,x1,y1,x2,y2],...]
  average_precison = []
  epsilon = 1e-6
  for c in range(num_classes):
    detections =[]
    ground_truths=[]
    for detection in pre_boxes:
      if detection[1] == c:
        detections.append(detection)
    for true_box in true_boxes:
      if true_box[1] == c:
        ground_truths.append(true_box)
    # note that the list detections hold all the prediction boxes and ground truth hold actual box in one big image that cotent small image like cat, car

    #img 0 has 3boxes
    # img 1 has 5 boxes
    amount_bboxes = Counter([gt[0] for gt in ground_truths])
    # amount_boxes = {0:1,1:5}
    for key,val in amount_bboxes.items():
      amount_bboxes[key] = torch.zeros(val)
      # amount_boxes = {0:torch.tensor([0,0,0]),1:torch.tensor([0,0,0,0,0])}
      # we doing this to prevent multiple prediction bboxes appear in 1 actual box, only the first one will be mark as TP
       #and rest of the predicted boxes on the same obj will be count as FP
    detections.sort(key= lambda x: x[2], reverse = True)
    TP = torch.zeros((len(detections)))
    FP = torch.zeros((len(detections)))
    total_true_boxes = len(ground_truths)

    for dectection_idx, dectection in enumerate(detections):
      #filter out all other actual boxes from the dataset, only remain the actual boxes that we want to use to compare with the prediction box, let say we want to test prediction
      #box 1, so we need to filter out all the actual box that is not 1
      ground_truth_img = [boxes for boxes in ground_truths if boxes[0] == dectection[0]]
      num_gts = len(ground_truth_img)
      best_iou = 0
      for idx, gt in enumerate(ground_truth_img):
        iou = IOU(torch.tensor(dectection[3:]),torch.tensor(gt[3:]),box_format=box_format)
        if iou > best_iou:
          best_iou = iou
          best_gt_idx = idx
        # after this, we have all the prediction bboxes for one actual boxes, however, since one actual box can one have one prediction box, we need to compare all of them to
        # find out the one with the best IOU
      if best_iou > IOU_threshold:
        if amount_bboxes[dectection[0]][best_gt_idx] == 0:
          TP[dectection_idx] = 1
          amount_bboxes[dectection[0]][best_gt_idx] = 1
        else:
          FP[dectection_idx] = 1
      else:
        FP[dectection_idx] = 1
    TP_cumsum = torch.cumsum(TP, dim=0)
    FP_cumsum = torch.cumsum(FP, dim=0)
    recalls = TP_cumsum / (total_true_boxes + epsilon)
    precisions = torch.divide(TP_cumsum, (TP_cumsum + FP_cumsum + epsilon))
    precisions = torch.cat((torch.tensor([1.]), precisions)) # Changed 1 to 1.
    recalls = torch.cat((torch.tensor([0.]), recalls)) # Changed 0 to 0.
    # torch.trapz for numerical integration
    average_precison.append(torch.trapz(precisions, recalls))

  return sum(average_precison) / len(average_precison)

In [ ]:
b1 = torch.tensor([[5.0, 5.0, 4.0, 4.0]])  # (x, y, w, h)
b2 = torch.tensor([[6.0, 6.0, 4.0, 4.0]])

print(IOU(b1, b2, box_format="midpoint"))

In [ ]:
# implement max_min suppression
# theory of mns:
# this algo is aimed to remove repeated bouding box by comparing the IOU value between 2 bboxes to a hyperparameter (threshold) given by the trainer
# given 3 bboxes, this mns will calculate the IOU between 2 boxes, if the value of IOU is larger than the threshold , the boxes is removed.
# it difference from noraml IOU, mns aim to sort all the high confident boxes, it will remove high confident but repeated box
def non_max_supression(bboxes, IOU_threshold, threshold, box_format="conner"):
  assert type(bboxes) == list #a simple check if the bboxes is a list or not

  temp_box = []
  for box in bboxes:
    if box[1] > threshold:
      temp_box.append(box)
      #looking for all the confident box that above the threshold
  bboxes = temp_box
  bboxes_after_mns =[]
  bboxes = sorted(bboxes, key =lambda x: x[1], reverse = True) # sort the box with the highest to the first
  #make a loop the sort by requirement
  while bboxes:
    chosen_box = bboxes.pop(0) # first we take the first box in the list aka the highest box confident boxes[0]

    temp_box_2=[]
    for box in bboxes:
      if box[0] != chosen_box[0] or IOU(torch.tensor(chosen_box[2:]),torch.tensor(box[2:]), box_format = box_format) < IOU_threshold:
        temp_box_2.append(box)
        # now we loop through, and keep box that: fit either not the same label or did not repeated

    bboxes = temp_box_2
    # after this line, bboxes will only remain either the box with a difference label with the first box that just being pop out or the same label but not repeated
    bboxes_after_mns.append(chosen_box)
    # now we save that chosen_box in a list since the choosen box will be renew each time when the while started
    # the next while loop with boxes, it will pop boxes [1] in the first while loop aka boxes[0] at this time, if this box is label difference, we will restart the mns process
    # or in the other case, it will find if there is other high but smaller confident and repeated box, ensure that an object have enough boxes with high confident (no repeated)
  return bboxes_after_mns
  # rebuild IOU again next time



In [ ]:
def get_bboxes(
    loader,
    model,
    iou_threshold,
    threshold,
    pred_format="cells",
    box_format="midpoint",
    device="cuda",
):
    all_pred_boxes = []
    all_true_boxes = []

    # make sure model is in eval before get bboxes
    model.eval()
    train_idx = 0

    for batch_idx, (x, labels) in enumerate(loader):
        x = x.to(device)
        labels = labels.to(device)

        with torch.no_grad():
            predictions = model(x)

        batch_size = x.shape[0]
        true_bboxes = cellboxes_to_boxes(labels)
        bboxes = cellboxes_to_boxes(predictions)

        for idx in range(batch_size):
            nms_boxes = non_max_supression(
                bboxes[idx],
                IOU_threshold=iou_threshold,
                threshold=threshold,
                box_format=box_format,
            )


            #if batch_idx == 0 and idx == 0:
            #    plot_image(x[idx].permute(1,2,0).to("cpu"), nms_boxes)
            #    print(nms_boxes)

            for nms_box in nms_boxes:
                all_pred_boxes.append([train_idx] + nms_box)

            for box in true_bboxes[idx]:
                # many will get converted to 0 pred
                if box[1] > threshold:
                    all_true_boxes.append([train_idx] + box)

            train_idx += 1

    model.train()
    return all_pred_boxes, all_true_boxes



def convert_cellboxes(predictions, S=7):
    """
    Converts bounding boxes output from Yolo with
    an image split size of S into entire image ratios
    rather than relative to cell ratios. Tried to do this
    vectorized, but this resulted in quite difficult to read
    code... Use as a black box? Or implement a more intuitive,
    using 2 for loops iterating range(S) and convert them one
    by one, resulting in a slower but more readable implementation.
    """

    predictions = predictions.to("cpu")
    batch_size = predictions.shape[0]
    predictions = predictions.reshape(batch_size, 7, 7, 30)
    bboxes1 = predictions[..., 21:25]
    bboxes2 = predictions[..., 26:30]
    scores = torch.cat(
        (predictions[..., 20].unsqueeze(0), predictions[..., 25].unsqueeze(0)), dim=0
    )
    best_box = scores.argmax(0).unsqueeze(-1)
    best_boxes = bboxes1 * (1 - best_box) + best_box * bboxes2
    cell_indices = torch.arange(7).repeat(batch_size, 7, 1).unsqueeze(-1)
    x = 1 / S * (best_boxes[..., :1] + cell_indices)
    y = 1 / S * (best_boxes[..., 1:2] + cell_indices.permute(0, 2, 1, 3))
    w_y = 1 / S * best_boxes[..., 2:4]
    converted_bboxes = torch.cat((x, y, w_y), dim=-1)
    predicted_class = predictions[..., :20].argmax(-1).unsqueeze(-1)
    best_confidence = torch.max(predictions[..., 20], predictions[..., 25]).unsqueeze(
        -1
    )
    converted_preds = torch.cat(
        (predicted_class, best_confidence, converted_bboxes), dim=-1
    )

def cellboxes_to_boxes(out, S=7):
    converted_pred = convert_cellboxes(out).reshape(out.shape[0], S * S, -1)
    converted_pred[..., 0] = converted_pred[..., 0].long()
    all_bboxes = []

    for ex_idx in range(out.shape[0]):
        bboxes = []

        for bbox_idx in range(S * S):
            bboxes.append([x.item() for x in converted_pred[ex_idx, bbox_idx, :]])
        all_bboxes.append(bboxes)

    return all_bboxes


In [ ]:
from torch import nn
class YoloNN(nn.Module):
  def __init__(self):
    super().__init__()
    self.Block1 = nn.Sequential(
        nn.Conv2d(in_channels=3,out_channels=64,kernel_size=7,stride=2,padding=3),
        nn.BatchNorm2d(64),
        nn.LeakyReLU(0.1,inplace=True),
        nn.MaxPool2d(kernel_size=2,stride=2),
        nn.Conv2d(in_channels=64,out_channels=192,kernel_size=3,stride=1,padding=1),
        nn.BatchNorm2d(192),
        nn.LeakyReLU(0.1,inplace=True),
        nn.MaxPool2d(kernel_size=2,stride=2),
        nn.Conv2d(in_channels=192,out_channels=128,kernel_size=1,stride=1,padding=0),
        nn.BatchNorm2d(128),
        nn.LeakyReLU(0.1,inplace=True),
        nn.Conv2d(in_channels=128,out_channels=256,kernel_size=3,stride=1,padding=1),
        nn.BatchNorm2d(256),
        nn.LeakyReLU(0.1,inplace=True),
        nn.Conv2d(in_channels=256,out_channels=256,kernel_size=1,stride=1,padding=0),
        nn.BatchNorm2d(256),
        nn.LeakyReLU(0.1,inplace=True),
        nn.Conv2d(in_channels=256,out_channels=512,kernel_size=3,stride=1,padding=1),
        nn.BatchNorm2d(512),
        nn.LeakyReLU(0.1,inplace=True),
        nn.MaxPool2d(kernel_size=2,stride=2),

        nn.Conv2d(in_channels=512,out_channels=256,kernel_size=1,stride=1,padding=0),
        nn.BatchNorm2d(256),
        nn.LeakyReLU(0.1,inplace=True),
        nn.Conv2d(in_channels=256,out_channels=512,kernel_size=3,stride=1,padding=1),
        nn.BatchNorm2d(512),
        nn.LeakyReLU(0.1,inplace=True),
        nn.Conv2d(in_channels=512,out_channels=256,kernel_size=1,stride=1,padding=0),
        nn.BatchNorm2d(256),
        nn.LeakyReLU(0.1,inplace=True),
        nn.Conv2d(in_channels=256,out_channels=512,kernel_size=3,stride=1,padding=1),
        nn.BatchNorm2d(512),
        nn.LeakyReLU(0.1,inplace=True),
        nn.Conv2d(in_channels=512,out_channels=256,kernel_size=1,stride=1,padding=0),
        nn.BatchNorm2d(256),
        nn.LeakyReLU(0.1,inplace=True),
        nn.Conv2d(in_channels=256,out_channels=512,kernel_size=3,stride=1,padding=1),
        nn.BatchNorm2d(512),
        nn.LeakyReLU(0.1,inplace=True),
        nn.Conv2d(in_channels=512,out_channels=256,kernel_size=1,stride=1,padding=0),
        nn.BatchNorm2d(256),
        nn.LeakyReLU(0.1,inplace=True),
        nn.Conv2d(in_channels=256,out_channels=512,kernel_size=3,stride=1,padding=1),
        nn.BatchNorm2d(512),
        nn.LeakyReLU(0.1,inplace=True),

        nn.Conv2d(in_channels=512,out_channels=512,kernel_size=1,stride=1,padding=0),
        nn.BatchNorm2d(512),
        nn.LeakyReLU(0.1,inplace=True),
        nn.Conv2d(in_channels=512,out_channels=1024,kernel_size=3,stride=1,padding=1),
        nn.BatchNorm2d(1024),
        nn.LeakyReLU(0.1,inplace=True),
        nn.MaxPool2d(kernel_size=2,stride=2),

        nn.Conv2d(in_channels=1024,out_channels=512,kernel_size=1,stride=1,padding=0),
        nn.BatchNorm2d(512),
        nn.LeakyReLU(0.1,inplace=True),
        nn.Conv2d(in_channels=512,out_channels=1024,kernel_size=3,stride=1,padding=1),
        nn.BatchNorm2d(1024),
        nn.LeakyReLU(0.1,inplace=True),
        nn.Conv2d(in_channels=1024,out_channels=512,kernel_size=1,stride=1,padding=0),
        nn.BatchNorm2d(512),
        nn.LeakyReLU(0.1,inplace=True),
        nn.Conv2d(in_channels=512,out_channels=1024,kernel_size=3,stride=1,padding=1),
        nn.BatchNorm2d(1024),
        nn.LeakyReLU(0.1,inplace=True),

        nn.Conv2d(in_channels=1024,out_channels=1024,kernel_size=3,stride=1,padding=1),
        nn.BatchNorm2d(1024),
        nn.LeakyReLU(0.1,inplace=True),
        nn.Conv2d(in_channels=1024,out_channels=1024,kernel_size=3,stride=2,padding=1),
        nn.BatchNorm2d(1024),
        nn.LeakyReLU(0.1,inplace=True),

        nn.Conv2d(in_channels=1024,out_channels=1024,kernel_size=3,stride=1,padding=1),
        nn.BatchNorm2d(1024),
        nn.LeakyReLU(0.1,inplace=True),
        nn.Conv2d(in_channels=1024,out_channels=1024,kernel_size=3,stride=1,padding=1),
        nn.BatchNorm2d(1024),
        nn.LeakyReLU(0.1,inplace=True),
        nn.Flatten(),
        nn.Linear(1024*7*7,4096),
        nn.Linear(4096,7*7*30)
    )
  def forward(self,x):
    x = self.Block1(x)
    return x




In [ ]:

import torch
module=YoloNN()
tensorss = torch.rand(3, 448,448)
tensorss = tensorss.unsqueeze(0)
print(tensorss.shape)

In [ ]:
x = module(tensorss)
print(x.shape)

In [ ]:
import torch.nn as nn
architecture_config = [
    # tuple: (kernel_size, num_filters, stride, padding)
    (7, 64, 2, 3),
    "M",
    (3, 192, 1, 1),
    "M",
    (1, 128, 1, 0),
    (3, 256, 1, 1),
    (1, 256, 1, 0),
    (3, 512, 1, 1),
    "M",
    # List
    [(1, 256, 1, 0), (3, 512, 1, 1), 4],
    (1, 512, 1, 0),
    (3, 1024, 1, 1),
    "M",
    [(1, 512, 1, 0), (3, 1024, 1, 1), 2],
    (3, 1024, 1, 1),
    (3, 1024, 1, 1),
    (3, 1024, 1, 1),
    (3, 1024, 1, 1),
]



In [ ]:
class CNNBlock(nn.Module):
  def __init__(self,in_channels,out_channels,**kwargs):
    super(CNNBlock, self).__init__()
    self.conv = nn.Conv2d(in_channels,out_channels,bias=False,**kwargs)
    self.batchnorm = nn.BatchNorm2d(out_channels)
    self.leakyrelu = nn.LeakyReLU(0.1)

  def forward(self,x):
    return self.leakyrelu(self.batchnorm(self.conv(x)))


In [ ]:
class Yolov1(nn.Module):
  def __init__(self,in_channels=3,**kwargs):
    super(Yolov1, self).__init__()
    self.architecture = architecture_config
    self.in_channels = in_channels
    self.darknet = self._create_conv_layers(self.architecture)
    self.fcs = self._create_fcs(**kwargs)

  def forward(self,x):
      x = self.darknet(x)
      return self.fcs(torch.flatten(x,start_dim=1))

  def _create_conv_layers(self,architecture):


      layer =[]
      in_channels = self.in_channels

      for x in architecture:
        if type(x) == tuple:
          layer += [
              CNNBlock(in_channels, x[1], kernel_size=x[0], stride=x[2], padding=x[3])
          ]
          in_channels = x[1]
        elif type(x) == str:
          layer += [nn.MaxPool2d(kernel_size=2,stride=2)]
        elif type(x) == list:
          conv1 = x[0]# tuple
          conv2 = x[1]# tuple
          num_repeats = x[2] #int

          for _ in range(num_repeats):
            layer += [
                CNNBlock(in_channels,
                         conv1[1],
                         kernel_size=conv1[0],
                         stride=conv1[2], padding=conv1[3])
            ]
            layer += [
                CNNBlock(conv1[1],
                         conv2[1],
                         kernel_size=conv2[0],
                         stride=conv2[2], padding=conv2[3])
            ]
            in_channels = conv2[1]
      return nn.Sequential(*layer)

  def _create_fcs(self,split_size,num_boxes,num_classes):
        S,B,C = split_size,num_boxes,num_classes
        return nn.Sequential(
            nn.Flatten(),
            nn.Linear(1024*14*14,496), #Orginal paper this should be 4096
            nn.Dropout(0.0),
            nn.LeakyReLU(0.1),
            nn.Linear(496,S*S*(C+B*5)) # (S,S,30) where C+B * 5 =30
        )



In [ ]:
def test(S=7,B=2,C=20):
  model= Yolov1(split_size=S,num_boxes=B,num_classes=C)
  x= torch.randn((2,3,448,448))
  print(model(x).shape)
test()

In [ ]:
class Yololoss(nn.Module):
  def __init__(self,S=7,B=2,C=20):
    super(Yololoss, self).__init__()
    self.mse = nn.MSELoss(reduction="sum")
    self.S = S
    self.B = B
    self.C = C
    self.lambda_noobj =0.5
    self.lambda_coord = 5
  def forward(self, predictions, target):
    predictions = predictions.reshape(-1, self.S, self.S, self.C +self.B*5)

    iou_b1 = IOU(predictions[...,21:25], target[...,21:25], box_format="midpoint") # Added box_format
    iou_b2 = IOU(predictions[...,26:30], target[...,21:25], box_format="midpoint") # Added box_format
    ious = torch.cat([iou_b1.unsqueeze(0), iou_b2.unsqueeze(0)],dim=0)
    iou_max,best_box = torch.max(ious,dim=0)
    exists_box = target[...,20].unsqueeze(3) #identity of object I, is there any obj in box I


    # Box coordinated
    box_predictions = exists_box * (
        best_box * predictions[...,26:30] + (1-best_box) * predictions[...,21:25]
        # now we need to find which box perform better to use, best box will be 1 if the second box b2 perfrom better and 0 if the first box did better
        # but first we always check if there is a bouding box first, by multiple by exist box, this value will be 0 if there is no bouding box

    )

    box_target = exists_box * target[...,21:25]
    #now, we get the target infor, check if there is any box first then multiple with the coor of target

    # Fix: Avoid in-place operation on tensor requiring gradients
    box_predictions_xy = box_predictions[..., :2]
    box_predictions_wh_transformed = torch.sign(box_predictions[..., 2:4]) * torch.sqrt(
        torch.abs(box_predictions[..., 2:4]) + 1e-6
    )
    box_predictions_transformed = torch.cat((box_predictions_xy, box_predictions_wh_transformed), dim=-1)

    # Also apply the same non-in-place transformation for consistency with target
    box_target_xy = box_target[..., :2]
    box_target_wh_transformed = torch.sqrt(box_target[..., 2:4])
    box_target_transformed = torch.cat((box_target_xy, box_target_wh_transformed), dim=-1)


    # now we will cal the mse error by plug-in data
    box_loss = self.mse(
        torch.flatten(box_predictions_transformed,end_dim=-2),
        torch.flatten(box_target_transformed,end_dim=-2)
        # end_dim = -2 because mse only except the data shaped in 2d
        # if our we have many boxes (N,S,S,4), it will become (N*S*S,4)
    )

    #Object loss
    # note: best_box currently will hold the highest IOU value
    # THE code here is to checkw which bouding boxes among the boxes which the highest
    # IOU is responsible
    #x1, y1, w1, h1, conf1,   <- first box (starts at index 20, if classes take up 20 slots)
    #x2, y2, w2, h2, conf2,   <- second box (starts at index 25)
    # it can be understand like an if statement where depend on which boxes is the best box
    # bouding box 1 or 2, it will take the conf value of  that box
    # the reason why the index of bouding box value start at index 20:21 is because the form 1-16 holding class prediction, 16-20 holding boxes paramneter
    pred_box = best_box * predictions[...,25:26] + (1 - best_box) * predictions[...,20:21]


    #object loss will also have to be flatten due to it need to be the same as boxes dimension
    # (N*S*S,1)
    # exist box == 0 if there is no obj, 1 otherwise
    # Summary: This line calculates how close the predicted confidence is to the actual object presence (1 or 0), only in cells that contain objects.
    object_loss = self.mse(torch.flatten(exists_box * pred_box)
    ,torch.flatten(exists_box * target[...,20:21])) # Fixed typo from 20.21 to 20:21


    # if there is no obj loss
    # (N,S,S,1) -> (N, S*S)
    # this part will make the value of target[...,20:21] become 0 if there is no obj
    no_object_loss = self.mse(torch.flatten((1-exists_box) * predictions[...,20:21], start_dim=1),
                              torch.flatten((1-exists_box) * target[...,20:21], start_dim=1))

    no_object_loss += self.mse(torch.flatten((1-exists_box) * predictions[...,25:26], start_dim=1),
                              torch.flatten((1-exists_box) * target[...,20:21], start_dim=1))

    # FOR CLASS LOSS

    # (N,S,S,20) -> (N*S*S,20)
    # this part will take the bouding box value prediction (only take the one that have obj)
    # and compare with the ground value, then take the mse of it
    class_loss = self.mse(
        torch.flatten(exists_box * predictions[...,:20], end_dim = -2),
        torch.flatten(exists_box * target[...,:20],end_dim = -2)
    )

    # take all the loss data
    loss = (
        self.lambda_coord * box_loss
        + object_loss
        + self.lambda_noobj * no_object_loss
        +class_loss
    )

    return loss


In [ ]:
# modifile dataset
from google.colab import drive
drive.mount('/content/drive')
import torch
import os
import pandas as pd
from PIL import Image

#creating custom dataset loading
class VOCDataset(torch.utils.data.Dataset):
  def __init__(self, csv_file, img_dir, label_dir, S=7,B=2,C=20, transform =None):
    self.annotation = pd.read_csv(csv_file)
    self.img_dir = img_dir
    self.label_dir = label_dir

    self.transform = transform
    self.S = S
    self.B =B
    self.C = C
  def __len__(self):
    return len(self.annotation)


  def __getitem__(self, index):
    label_path = os.path.join(self.label_dir, self.annotation.iloc[index, 1])
    boxes = []
    # this part modifiling the file, assume that the label is always int, but the height and witdth and other are likely to be float, since we need float so we do some transition
    with open(label_path) as f:
      for label in f.readlines():
        class_label,x,y,width,heigth =[
            float(x) if float(x) != int(float(x)) else int(x)
            for x in label.replace("\n","").split()
        ]
        boxes.append([class_label,x,y,width,heigth])
    #open the file and add the modifiled into the tensor
    img_path = os.path.join(self.img_dir, self.annotation.iloc[index, 0])
    image = Image.open(img_path)
    boxes = torch.tensor(boxes)

    if self.transform:
      image, boxes= self.transform(image, boxes)

    label_matrix = torch.zeros((self.S, self.S, self.C+5*self.B))
    for box in boxes:
      # then somehow we convert back to the list
      class_label,x,y,width, heigth = box.tolist()
      class_label = int(class_label)
      i, j = int(self.S *y), int(self.S *x)
      x_cell, y_cell =self.S*x -j, self.S*y-i
      width_cell, height_cell =(
          width *self.S, heigth * self.S
      )

      if label_matrix[i,j,20] ==0:
        label_matrix [i,j,20] =1
        box_coordinates = torch.tensor(
            [x_cell,y_cell,width_cell,height_cell]
        )
        label_matrix[i,j,21:25] = box_coordinates
        label_matrix[i,j,class_label] =1
    return image, label_matrix


In [ ]:
# Set your dataset paths
csv_file = "/content/drive/MyDrive/VOC_Dataset/archive/100examples.csv"
img_dir =  "/content/drive/MyDrive/VOC_Dataset/archive/images"
label_dir = "/content/drive/MyDrive/VOC_Dataset/archive/labels"

# Create the dataset object
dataset = VOCDataset(csv_file, img_dir, label_dir)

# Optional: Test it
image, label_matrix = dataset[1]
print("Image size:", image.size)
print("Label matrix shape:", label_matrix.shape)
#note I did not upload the whole image and label file sicen it too heavy

In [ ]:
seed = 123
torch.manual_seed(seed)
import torch
import torchvision.transforms as transforms
import torch.optim as optim
import torchvision.transforms.functional as FT
from tqdm import tqdm
from torch.utils.data import DataLoader
#Hyperparameter ect
LEARNING_RATE = 2e-5
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 16
WEIGHT_DECAY =0
EPOCHS = 100
NUM_WORKERS = 2
PIN_MEMORY = True
LOAD_MODEL = True
LOAD_MODEL_FILE = "overfit.pth.tar"
IMG_DIR ="/content/drive/MyDrive/VOC_Dataset/archive/images"
LABEL_DIR = "/content/drive/MyDrive/VOC_Dataset/archive/labels"
#make sure that they are resize,
class Compose(object):
  def __init__(self, transforms):
    self.transforms = transforms


  def __call__(self, img, bboxes):
    for t in self.transforms:
      img, bboxes = t(img),bboxes
    return img,bboxes
transform = Compose([transforms.Resize((448,448)), transforms.ToTensor()])
# function to create bar to check the progress
def train_fn(train_loader,model,optimizer,loss_fn):
  loop = tqdm(train_loader,leave= True)
  mean_loss =[]

  for batch_idx, (x,y) in enumerate(loop):
    x,y = x.to(DEVICE), y.to(DEVICE)
    out = model(x)
    loss = loss_fn(out,y)
    mean_loss.append(loss.item())
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    #now we will upgrade the progress bar
    loop.set_postfix(loss = loss.item())
  print("The mean loss was: "+ str(sum(mean_loss)/len(mean_loss))) # Converted to string
# the actual main todo stuff
def main():
  model= Yolov1(split_size =7, num_boxes=2,num_classes=20).to(DEVICE)
  optimizer= optim.Adam(model.parameters(),lr=LEARNING_RATE,weight_decay=WEIGHT_DECAY)
  loss_fn = Yololoss()

  if LOAD_MODEL:
    # Only load checkpoint if file exists to avoid error on first run
    if os.path.exists(LOAD_MODEL_FILE):
        load_checkpoint(torch.load(LOAD_MODEL_FILE), model, optimizer)

    train_dataset = VOCDataset(
        "/content/drive/MyDrive/VOC_Dataset/archive/100examples.csv",
        transform=transform,
        img_dir=IMG_DIR,
        label_dir=LABEL_DIR,
    )

    test_dataset = VOCDataset(
        "/content/drive/MyDrive/VOC_Dataset/archive/test.csv", transform=transform, img_dir=IMG_DIR, label_dir=LABEL_DIR,
    )

    train_loader = DataLoader(
        dataset=train_dataset,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        shuffle=True,
        drop_last=True,
    )

    test_loader = DataLoader(
        dataset=test_dataset,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        shuffle=True,
        drop_last=True,
    )

    for epoch in range(EPOCHS):
      """
      for x, y in train_loader:
            x = x.to(DEVICE)
            for idx in range(30):
                bboxes = cellboxes_to_boxes(model(x))
                bboxes = non_max_suppression(bboxes[idx], iou_threshold=0.5, threshold=0.4, box_format="midpoint")
                plot_image(x[idx].permute(1,2,0).to("cpu"), bboxes)

            import sys
            sys.exit()
      """

      pre_boxes,target_boxes = get_bboxes(train_loader,model,iou_threshold=0.5, threshold=0.4)

      mean_average_precision = mean_average_presision(pre_boxes,target_boxes,IOU_threshold=0.5,box_format="midpoint") # Fixed typo from pre to precision and box_fornat to box_format

      print("train map: "+ str(mean_average_precision)) # Converted to string

      train_fn(train_loader,model,optimizer,loss_fn)

In [ ]:
main()